# Prerequisite 01 · Python：读懂训练循环

[预备篇导航](Prerequisite00_Index.ipynb) · [下一章：NumPy](Prerequisite02_NumPy.ipynb)

**目标：**能够阅读变量、容器、条件、循环、函数和类；能够把一段 agent–environment 交互拆成小函数。这里不讲与 RL 无关的高级语法。


## 1. 数据：一个 transition 就是一条记录

Python 常用容器在 RL 中各有职责：tuple 表示固定字段，list 收集轨迹，dict 保存配置或指标。


In [1]:
transition = ("state_0", 1, 0.5, "state_1", False)
state, action, reward, next_state, done = transition
trajectory = [transition]
config = {"gamma": 0.99, "learning_rate": 1e-3}

print("action/reward:", action, reward)
print("trajectory length:", len(trajectory))
print("gamma:", config["gamma"])


action/reward: 1 0.5
trajectory length: 1
gamma: 0.99


## 2. 控制流与函数

训练循环本质上是：重复选动作、执行、保存经验，遇到终止就 reset。函数应只有清楚的输入和输出，避免依赖难追踪的全局变量。


In [2]:
def discounted_return(rewards, gamma):
    total = 0.0
    for reward in reversed(rewards):
        total = reward + gamma * total
    return total

rewards = [0.0, 0.0, 1.0]
print("return:", discounted_return(rewards, gamma=0.9))
assert abs(discounted_return(rewards, 0.9) - 0.81) < 1e-9


return: 0.81


## 3. 类：把状态和行为放在一起

环境和神经网络通常是对象。`__init__` 保存状态，方法读取或修改它；不要把“类”理解为新算法。


In [3]:
class EpsilonGreedyAgent:
    def __init__(self, epsilon=0.1):
        self.epsilon = epsilon
        self.steps = 0

    def act(self, q_values):
        self.steps += 1
        # 为便于演示，这里只写 greedy 分支。
        return max(range(len(q_values)), key=lambda i: q_values[i])

agent = EpsilonGreedyAgent()
print("chosen action:", agent.act([0.2, 0.8, 0.1]), "steps:", agent.steps)


chosen action: 1 steps: 1


## 4. 初学者常见坑

- `=` 是赋值，`==` 是比较。
- list 是可变对象；复制时注意 `copy()` 与共享引用。
- 函数没有显式 `return` 时返回 `None`。
- 不要用 `from module import *`，变量来源会变得不清楚。
- 异常信息应从最后一行开始读，再沿 traceback 找到自己的代码。

**练习：**让 `discounted_return` 同时返回每个时刻的 return，例如 `[0.81, 0.9, 1.0]`。

<details><summary>参考思路</summary>反向遍历 reward，每次把新的 `total` 插到列表开头，或最后再 reverse。</details>


## 官方选读

- [Python 中文官方教程](https://docs.python.org/zh-cn/3/tutorial/index.html)：重点看控制流、数据结构、函数、模块和类。

**过关：**能不看答案写出 `discounted_return`，并能解释 tuple、list、dict 在训练代码中的不同用途。
